In [20]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)

In [21]:
for root, dirs, files in os.walk("."):
    if root.count(os.sep) > 3:
        continue
    for f in files:
        if f.endswith((".csv", ".npz", ".npy")):
            print(os.path.join(root, f))

.\test_features.csv
.\test_features.npz
.\train.csv
.\train.npz
.\outputs\task2_1_predictions.csv
.\outputs\task2_2_predictions.csv


In [22]:

TRAIN_PATH = "train.csv"                 
TEST_PATH  = "test_features.csv"     

assert os.path.exists(TRAIN_PATH), f"Не найден {TRAIN_PATH}"
assert os.path.exists(TEST_PATH),  f"Не найден {TEST_PATH}"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print("train shape:", df_train.shape)
print("test  shape:", df_test.shape)


train shape: (4000, 142)
test  shape: (1000, 141)


In [23]:
label_candidates = ["label", "prediction", "target", "y", "class"]
label_col = next((c for c in label_candidates if c in df_train.columns), None)

if label_col is None:
    raise ValueError(
        f"Кандидаты: {label_candidates}. "
        f"Есть колонки: {list(df_train.columns)}"
    )

print("Колонка метки:", label_col)
print("Распределение классов:")
print(df_train[label_col].value_counts().sort_index())

# --- определяем признаки t_* ---
feature_cols = [c for c in df_test.columns if c.startswith("t_")]
print(len(feature_cols))

missing = [c for c in feature_cols if c not in df_train.columns]


Колонка метки: prediction
Распределение классов:
prediction
0    2335
1    1414
2      77
3     155
4      19
Name: count, dtype: int64
140


In [24]:
common_cols = [c for c in feature_cols if c in df_train.columns]
print(len(common_cols))

X_train = df_train[common_cols].values.astype(np.float32)
y_train = df_train[label_col].values.astype(int)

X_test     = df_test[common_cols].values.astype(np.float32)
test_ids   = df_test["heartbeat_id"].values

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("Уникальные классы в y_train:", np.unique(y_train))

print("NaN в X_train:", np.isnan(X_train).sum(),
      "| NaN в X_test:", np.isnan(X_test).sum())

140
X_train: (4000, 140)  y_train: (4000,)
X_test : (1000, 140)
Уникальные классы в y_train: [0 1 2 3 4]
NaN в X_train: 0 | NaN в X_test: 0


In [25]:
pipe = Pipeline([   
    ("scaler",  StandardScaler()),
    ("clf",     RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

class_counts = pd.Series(y_train).value_counts()
min_count = int(class_counts.min())
n_splits = int(min(5, min_count))
print(f"Мин. кол-во образцов в классе: {min_count} -> n_splits={n_splits}")

if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
else:
    print(" Слишком мало данных ")

pipe.fit(X_train, y_train)


Мин. кол-во образцов в классе: 19 -> n_splits=5
CV accuracy: 0.9483 ± 0.0034


,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [26]:
y_train_pred = pipe.predict(X_train)
print("Train accuracy:", f"{np.mean(y_train_pred == y_train):.4f}")
print("\nClassification report (train):")
print(classification_report(y_train, y_train_pred, digits=4))

Train accuracy: 1.0000

Classification report (train):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      2335
           1     1.0000    1.0000    1.0000      1414
           2     1.0000    1.0000    1.0000        77
           3     1.0000    1.0000    1.0000       155
           4     1.0000    1.0000    1.0000        19

    accuracy                         1.0000      4000
   macro avg     1.0000    1.0000    1.0000      4000
weighted avg     1.0000    1.0000    1.0000      4000



In [27]:
y_pred = pipe.predict(X_test)

out = pd.DataFrame({
    "heartbeat_id": test_ids,
    "prediction":  y_pred.astype(int),
})

assert len(out) == len(df_test), "Число строк не совпадает с test_features.csv"
assert out["heartbeat_id"].is_unique, "heartbeat_id должен быть уникальным"
assert out["prediction"].notna().all(), "Есть NaN в prediction"
assert set(out["prediction"].unique()).issubset(set(range(5))), \
       f"Классы должны быть 0..4, а получены {out['prediction'].unique()}"

print(out.head())
print("\nРаспределение предсказаний:")
print(out["prediction"].value_counts().sort_index())

   heartbeat_id  prediction
0             2           0
1             5           0
2            12           0
3            17           0
4            20           0

Распределение предсказаний:
prediction
0    591
1    382
2      8
3     19
Name: count, dtype: int64


In [28]:
check = pd.read_csv("outputs/task2_1_predictions.csv")
assert list(check.columns) == ["heartbeat_id", "prediction"], "Колонки не те"
assert len(check) == len(df_test), "Число строк не совпадает"
assert check["heartbeat_id"].is_unique
assert check["prediction"].between(0, 4).all()
